# Transformer and supervised fine-tuning from first principles

This notebook builds a tiny causal language model and fine-tunes it on prompt/response pairs. It uses PyTorch tensors and autograd, but implements the Transformer attention calculation, causal mask, SFT label mask, loss, and generation loop directly.

## Learning objectives

By the end, you should be able to explain:

1. How token embeddings become contextual representations through causal self-attention.
2. Why a decoder-only model predicts one next token at every sequence position.
3. Why SFT is ordinary next-token cross-entropy applied to a curated dataset, usually with prompt-token losses masked out.
4. How teacher forcing during training differs from autoregressive generation at inference time.
5. What this toy example omits relative to real post-training.

## 1. The algorithm in one picture

For tokens $x_0, \ldots, x_{T-1}$, a causal language model estimates

$$p_\theta(x_1, \ldots, x_T) = \prod_{t=0}^{T-1} p_\theta(x_{t+1} \mid x_{\le t}).$$

A decoder-only Transformer repeatedly applies:

```text
token + position embeddings
        |
LayerNorm -> causal multi-head self-attention -> residual add
        |
LayerNorm -> position-wise MLP              -> residual add
        |
vocabulary logits for the next token at every position
```

SFT does not require a different architecture. It changes the examples and decides which next-token predictions contribute to the loss.

In [ ]:
from __future__ import annotations

import math
import random

import torch
from torch import nn
from torch.nn import functional as F

SEED = 7
random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)

print(f"PyTorch {torch.__version__}; device=cpu")

## 2. Turn supervised examples into language-model tokens

Real systems use a subword tokenizer and a chat template. A character vocabulary keeps this example transparent. Every record is serialized as:

```text
<bos> prompt <sep> response <eos>
```

The input is the sequence without its last token. The target is the same sequence shifted one token left. Prompt targets become `IGNORE_INDEX`, so only response tokens and `<eos>` contribute to SFT loss. This shift-and-mask order is a common source of off-by-one bugs.

In [ ]:
pairs = [
    ("question: 2+2?", "answer: 4"),
    ("question: 3+4?", "answer: 7"),
    ("opposite of hot?", "cold"),
    ("opposite of up?", "down"),
    ("say hello", "hello"),
    ("say goodbye", "goodbye"),
]

special_tokens = ["<pad>", "<bos>", "<sep>", "<eos>"]
characters = sorted(set("".join(prompt + response for prompt, response in pairs)))
vocabulary = special_tokens + characters
stoi = {token: index for index, token in enumerate(vocabulary)}
itos = {index: token for token, index in stoi.items()}
PAD, BOS, SEP, EOS = (stoi[token] for token in special_tokens)
IGNORE_INDEX = -100

def encode_text(text: str) -> list[int]:
    return [stoi[character] for character in text]

def readable(token_ids: list[int]) -> str:
    return " ".join(itos[token_id] for token_id in token_ids)

def make_training_example(prompt: str, response: str):
    sequence = [BOS] + encode_text(prompt) + [SEP] + encode_text(response) + [EOS]
    # A role flag follows each sequence token: False for prompt/template, True for response.
    response_role = (
        [False] * (1 + len(prompt) + 1)
        + [True] * (len(response) + 1)
    )
    inputs = sequence[:-1]
    full_targets = sequence[1:]
    response_targets = [
        target if is_response else IGNORE_INDEX
        for target, is_response in zip(full_targets, response_role[1:])
    ]
    return inputs, full_targets, response_targets

examples = [make_training_example(*pair) for pair in pairs]
max_length = max(len(inputs) for inputs, _, _ in examples)

def collate(examples):
    batch_size = len(examples)
    input_ids = torch.full((batch_size, max_length), PAD, dtype=torch.long)
    full_labels = torch.full((batch_size, max_length), IGNORE_INDEX, dtype=torch.long)
    sft_labels = torch.full((batch_size, max_length), IGNORE_INDEX, dtype=torch.long)
    for row, (inputs, full_targets, response_targets) in enumerate(examples):
        length = len(inputs)
        input_ids[row, :length] = torch.tensor(inputs)
        full_labels[row, :length] = torch.tensor(full_targets)
        sft_labels[row, :length] = torch.tensor(response_targets)
    return input_ids, full_labels, sft_labels

input_ids, full_labels, sft_labels = collate(examples)
attention_mask = input_ids.ne(PAD)

first_length = attention_mask[0].sum().item()
print("input:     ", readable(input_ids[0, :first_length].tolist()))
print("next token:", readable(full_labels[0, :first_length].tolist()))
print("SFT target:", " ".join(
    "---" if token_id == IGNORE_INDEX else itos[token_id]
    for token_id in sft_labels[0, :first_length].tolist()
))

## 3. Implement causal multi-head self-attention

For each head, learned projections produce queries, keys, and values:

$$Q=XW_Q, \quad K=XW_K, \quad V=XW_V.$$

Attention weights are

$$A = \operatorname{softmax}\left(\frac{QK^\top}{\sqrt{d_h}} + M\right), \qquad \operatorname{Attention}(X)=AV,$$

where the causal mask $M_{ij}=-\infty$ when key position $j$ is in the future of query position $i$. Multiple heads run this operation in parallel and concatenate their results.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, model_dim: int, num_heads: int, max_seq_len: int):
        super().__init__()
        assert model_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = model_dim // num_heads
        self.qkv = nn.Linear(model_dim, 3 * model_dim, bias=False)
        self.output = nn.Linear(model_dim, model_dim, bias=False)
        causal = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", causal, persistent=False)

    def forward(self, x: torch.Tensor, token_mask: torch.Tensor):
        batch_size, seq_len, model_dim = x.shape
        qkv = self.qkv(x)
        qkv = qkv.view(batch_size, seq_len, 3, self.num_heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4).unbind(dim=0)

        scores = q @ k.transpose(-2, -1) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(~self.causal_mask[:seq_len, :seq_len], float("-inf"))
        scores = scores.masked_fill(~token_mask[:, None, None, :], float("-inf"))
        weights = F.softmax(scores, dim=-1)

        context = weights @ v
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, model_dim)
        return self.output(context), weights


class TransformerBlock(nn.Module):
    def __init__(self, model_dim: int, num_heads: int, max_seq_len: int):
        super().__init__()
        self.attention_norm = nn.LayerNorm(model_dim)
        self.attention = CausalSelfAttention(model_dim, num_heads, max_seq_len)
        self.mlp_norm = nn.LayerNorm(model_dim)
        self.mlp = nn.Sequential(
            nn.Linear(model_dim, 4 * model_dim),
            nn.GELU(),
            nn.Linear(4 * model_dim, model_dim),
        )

    def forward(self, x: torch.Tensor, token_mask: torch.Tensor):
        attended, weights = self.attention(self.attention_norm(x), token_mask)
        x = x + attended
        x = x + self.mlp(self.mlp_norm(x))
        return x, weights


class TinyCausalLM(nn.Module):
    def __init__(self, vocab_size: int, max_seq_len: int, model_dim=48, num_heads=4, layers=2):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_embedding = nn.Embedding(vocab_size, model_dim)
        self.position_embedding = nn.Embedding(max_seq_len, model_dim)
        self.blocks = nn.ModuleList([
            TransformerBlock(model_dim, num_heads, max_seq_len) for _ in range(layers)
        ])
        self.final_norm = nn.LayerNorm(model_dim)
        self.lm_head = nn.Linear(model_dim, vocab_size, bias=False)
        self.lm_head.weight = self.token_embedding.weight  # common weight tying

    def forward(self, token_ids: torch.Tensor, token_mask: torch.Tensor | None = None):
        if token_mask is None:
            token_mask = token_ids.ne(PAD)
        positions = torch.arange(token_ids.shape[1], device=token_ids.device)
        x = self.token_embedding(token_ids) + self.position_embedding(positions)
        attention_maps = []
        for block in self.blocks:
            x, weights = block(x, token_mask)
            attention_maps.append(weights)
        return self.lm_head(self.final_norm(x)), attention_maps


model = TinyCausalLM(len(vocabulary), max_length)
parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(f"Vocabulary: {len(vocabulary)} tokens; model: {parameter_count:,} parameters")

In [ ]:
# Verify the defining causal invariant: no head can attend to a future position.
with torch.no_grad():
    _, attention_maps = model(input_ids, attention_mask)
first_head = attention_maps[0][0, 0]
future_attention = torch.triu(first_head, diagonal=1).abs().max().item()
print(f"Maximum attention weight assigned to the future: {future_attention:.1f}")
assert future_attention == 0.0

query_position = first_length - 1
valid_weights = first_head[query_position, :first_length]
top_weights, top_positions = valid_weights.topk(min(5, first_length))
print(f"At the last input token ({itos[input_ids[0, query_position].item()]}), head 0 attends most to:")
for weight, position in zip(top_weights.tolist(), top_positions.tolist()):
    print(f"  position {position:2d} token={itos[input_ids[0, position].item()]!r:8s} weight={weight:.3f}")

## 4. Define response-only SFT loss

Teacher forcing gives the model the ground-truth prefix in parallel and scores the correct next token. For a set $R$ of response-token positions, the SFT objective is

$$\mathcal{L}_{\mathrm{SFT}}(\theta) = -\frac{1}{|R|}\sum_{t \in R} \log p_\theta(x_{t+1} \mid x_{\le t}).$$

Framework cross-entropy can implement this directly, but spelling it out makes the mask semantics visible.

In [ ]:
def masked_next_token_loss(logits: torch.Tensor, labels: torch.Tensor):
    selected = labels.ne(IGNORE_INDEX)
    log_probabilities = F.log_softmax(logits, dim=-1)
    row, position = selected.nonzero(as_tuple=True)
    correct_token = labels[row, position]
    token_losses = -log_probabilities[row, position, correct_token]
    return token_losses.mean(), token_losses

logits, _ = model(input_ids, attention_mask)
initial_sft_loss, token_losses = masked_next_token_loss(logits, sft_labels)
print(f"Response tokens scored: {token_losses.numel()}")
print(f"Initial mean SFT loss: {initial_sft_loss.item():.3f}")

## 5. Teacher-forced training and autoregressive generation

Training computes every position in parallel because the causal mask prevents leakage. Generation cannot do that: it samples or selects one new token, appends it, and runs the model again. Production systems cache keys and values so earlier positions are not recomputed.

In [ ]:
@torch.no_grad()
def generate(model: TinyCausalLM, prompt: str, max_new_tokens=20) -> str:
    model.eval()
    generated = [BOS] + encode_text(prompt) + [SEP]
    response = []
    for _ in range(max_new_tokens):
        context = torch.tensor([generated[-model.max_seq_len:]], dtype=torch.long)
        logits, _ = model(context, torch.ones_like(context, dtype=torch.bool))
        next_token = logits[0, -1].argmax().item()
        if next_token == EOS:
            break
        generated.append(next_token)
        if next_token not in (PAD, BOS, SEP):
            response.append(itos[next_token])
    return "".join(response)

print("Before SFT:")
for prompt, expected in pairs[:3]:
    print(f"  {prompt!r} -> {generate(model, prompt)!r} (target {expected!r})")

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=0.01)
model.train()

for step in range(301):
    optimizer.zero_grad(set_to_none=True)
    logits, _ = model(input_ids, attention_mask)
    loss, _ = masked_next_token_loss(logits, sft_labels)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    if step % 50 == 0:
        print(f"step={step:3d}  loss={loss.item():.4f}")

print("\nAfter SFT:")
for prompt, expected in pairs:
    print(f"  {prompt!r} -> {generate(model, prompt)!r} (target {expected!r})")

The tiny model can memorize six examples. This confirms that the data pipeline, causal model, loss mask, backward pass, and generation loop are connected correctly. It does **not** demonstrate generalization. Real SFT starts from a pretrained model whose representations already encode language and broad capabilities.

## 6. Prompt masking changes the optimization signal

A full-sequence language-model loss also trains the model to reproduce prompts and template text. Response-only SFT conditions on those tokens but does not reward predicting them. Both objectives backpropagate through prompt representations because response predictions attend to the prompt; masking a prompt label does **not** remove the prompt from the computation graph.

In [ ]:
# Use a fresh model: the trained model has nearly zero response loss and gradients.
torch.manual_seed(SEED + 1)
gradient_probe = TinyCausalLM(len(vocabulary), max_length)

def selected_gradient_norms(labels: torch.Tensor) -> dict[str, float]:
    gradient_probe.zero_grad(set_to_none=True)
    logits, _ = gradient_probe(input_ids, attention_mask)
    loss, _ = masked_next_token_loss(logits, labels)
    loss.backward()
    return {
        "token embedding / tied head": gradient_probe.token_embedding.weight.grad.norm().item(),
        "first QKV projection": gradient_probe.blocks[0].attention.qkv.weight.grad.norm().item(),
        "first MLP input": gradient_probe.blocks[0].mlp[0].weight.grad.norm().item(),
    }

full_norms = selected_gradient_norms(full_labels)
sft_norms = selected_gradient_norms(sft_labels)
print(f"{'parameter group':30s} {'full LM':>10s} {'response SFT':>14s}")
for name in full_norms:
    print(f"{name:30s} {full_norms[name]:10.4f} {sft_norms[name]:14.4f}")

## 7. SFT algorithm summary

```text
given a pretrained causal LM and prompt/response records:
    serialize each record with the model's exact chat template
    tokenize and shift the sequence to make next-token labels
    replace non-response labels with IGNORE_INDEX

for each minibatch:
    logits = model(input_ids, causal_mask, padding_mask)
    loss = mean cross-entropy at non-ignored label positions
    backpropagate loss
    clip gradients if needed
    update parameters with AdamW or another optimizer
```

### What matters in real SFT

- **Starting checkpoint:** SFT shapes an existing capability distribution; training this toy model from scratch is only a mechanics demonstration.
- **Data quality and mixture:** examples define the behavior being cloned. Duplication, contradictions, style imbalance, and task mixture can dominate algorithm details.
- **Template correctness:** training and inference must agree on roles, separators, and special tokens.
- **Loss normalization:** token-mean loss gives longer responses more weight; example-mean alternatives answer a different objective.
- **Mask policy:** response-only, full-sequence, and multi-turn role masks produce different gradients.
- **Optimization scale:** learning rate, batch size, sequence packing, precision, and distributed reduction change stability and throughput.
- **Evaluation:** low training loss can mean memorization. Measure held-out task quality, regressions, formatting, safety, and calibration.

## 8. Exercises

1. Replace `sft_labels` with `full_labels` during training. Compare prompt reproduction, response learning, and gradient norms.
2. Remove the causal mask and rerun the invariant check. Explain why the training loss becomes invalid even if it decreases faster.
3. Change from four attention heads to one while keeping model width fixed. Track parameter count and learning behavior.
4. Add one contradictory response. Observe which answer the model produces and why dataset counts matter.
5. Add a long response and compare token-mean loss with a per-example normalized loss.
6. Split examples into train and held-out prompts to make the memorization/generalization distinction measurable.
7. Add a second user/assistant turn and construct a role mask that supervises assistant turns only.